# Feature Engineering & Tuned Ensembles for Regression

## Objective

The objective of this task is to improve the House Prices regression models through domain-driven feature engineering and properly tuned ensemble learning algorithms.

In this task, we will:

- Create meaningful features from existing housing information.
- Apply 5-fold cross-validation.
- Re-evaluate baseline regression models.
- Tune Random Forest and XGBoost using RandomizedSearchCV.
- Compare model performance using RMSE, R², and training time.

## Dataset

- Dataset: Ames Housing (House Prices)
- Target variable: SalePrice
- Problem type: Regression
- Validation strategy: 5-fold cross-validation
- Final evaluation: Untouched 20% holdout test set.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the cleaned House Prices dataset

df = pd.read_csv("train_cleaned.csv")

print("Dataset loaded successfully.")

print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Dataset shape: (1460, 86)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MoSold,YrSold,SaleType,SaleCondition,SalePrice,GrLivArea_Outlier,LotArea_Outlier,1stFlrSF_Outlier,TotalBsmtSF_Outlier,GarageArea_Outlier
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,2,2008,WD,Normal,208500,False,False,False,False,False
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,5,2007,WD,Normal,181500,False,False,False,False,False
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,9,2008,WD,Normal,223500,False,False,False,False,False
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,2,2006,WD,Abnorml,140000,False,False,False,False,False
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,12,2008,WD,Normal,250000,False,False,False,False,False


In [3]:
# Basic dataset inspection

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nDataset information:")
df.info()

print("\nMissing values:", df.isna().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSalePrice summary:")
display(df["SalePrice"].describe())

Number of rows: 1460
Number of columns: 86

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 86 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   1460 non-null   int64  
 1   MSSubClass           1460 non-null   int64  
 2   MSZoning             1460 non-null   object 
 3   LotFrontage          1460 non-null   float64
 4   LotArea              1460 non-null   int64  
 5   Street               1460 non-null   object 
 6   Alley                91 non-null     object 
 7   LotShape             1460 non-null   object 
 8   LandContour          1460 non-null   object 
 9   Utilities            1460 non-null   object 
 10  LotConfig            1460 non-null   object 
 11  LandSlope            1460 non-null   object 
 12  Neighborhood         1460 non-null   object 
 13  Condition1           1460 non-null   object 
 14  Condition2           14

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [4]:
# Remove target and unnecessary identifier/outlier columns

outlier_flags = [
    col for col in df.columns
    if col.endswith("_Outlier")
]

columns_to_drop = ["SalePrice", "Id"] + outlier_flags

X = df.drop(
    columns=columns_to_drop,
    errors="ignore"
)

y = df["SalePrice"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nRemoved outlier flags:", outlier_flags)
print("Id removed:", "Id" not in X.columns)
print("SalePrice excluded:", "SalePrice" not in X.columns)

Features shape: (1460, 79)
Target shape: (1460,)

Removed outlier flags: ['GrLivArea_Outlier', 'LotArea_Outlier', '1stFlrSF_Outlier', 'TotalBsmtSF_Outlier', 'GarageArea_Outlier']
Id removed: True
SalePrice excluded: True


In [5]:
# Split the dataset before preprocessing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining target mean:", round(y_train.mean(), 2))
print("Testing target mean:", round(y_test.mean(), 2))

assert len(X_train) + len(X_test) == len(df)
assert "SalePrice" not in X_train.columns

print("\nTrain-test split verified.")

Training rows: 1168
Testing rows: 292

Training target mean: 181441.54
Testing target mean: 178839.81

Train-test split verified.


In [6]:
# Create 5-fold cross-validation strategy

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation strategy created.")
print("Number of folds:", cv.get_n_splits())

Cross-validation strategy created.
Number of folds: 5


## Block 2: Feature Engineering

Feature engineering involves creating new meaningful variables from existing dataset columns.

For this task, five features will be created using domain knowledge about residential properties.

1. **TotalSF:** Combines basement, first-floor, and second-floor square footage to represent total living area.
2. **HouseAge:** Calculates the property's age at the time of sale using the year built and year sold.
3. **TotalBathrooms:** Combines full and half bathrooms, giving half bathrooms a weight of 0.5.
4. **HasRemodeled:** Indicates whether the house was remodeled after its original construction.
5. **TotalPorchSF:** Combines different porch and deck areas to represent outdoor space.

These features use information available before or at the time of sale and do not use SalePrice. Feature creation will be performed inside the machine-learning pipeline to prevent data leakage.

In [7]:
# Check columns required for feature engineering

required_columns = [
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "YearBuilt",
    "YrSold",
    "FullBath",
    "HalfBath",
    "BsmtFullBath",
    "BsmtHalfBath",
    "YearRemodAdd",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "WoodDeckSF"
]

missing_columns = [
    col for col in required_columns
    if col not in X.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All required columns are available.")

print("Number of required columns:", len(required_columns))

All required columns are available.
Number of required columns: 15


In [8]:
# Create a reusable feature engineering function

def engineer_features(data):

    data = data.copy()

    # 1. Total square footage
    data["TotalSF"] = (
        data["TotalBsmtSF"].fillna(0)
        + data["1stFlrSF"].fillna(0)
        + data["2ndFlrSF"].fillna(0)
    )

    # 2. House age at the time of sale
    data["HouseAge"] = (
        data["YrSold"] - data["YearBuilt"]
    )

    # 3. Total bathrooms
    data["TotalBathrooms"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    # 4. Whether the house was remodeled
    data["HasRemodeled"] = (
        data["YearRemodAdd"] > data["YearBuilt"]
    ).astype(int)

    # 5. Total porch and deck area
    data["TotalPorchSF"] = (
        data["OpenPorchSF"].fillna(0)
        + data["EnclosedPorch"].fillna(0)
        + data["3SsnPorch"].fillna(0)
        + data["ScreenPorch"].fillna(0)
        + data["WoodDeckSF"].fillna(0)
    )

    return data


print("Feature engineering function created successfully.")

Feature engineering function created successfully.


In [9]:
# Apply feature engineering

X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

print("Feature engineering applied successfully.")

print("Original training features:", X_train.shape[1])
print("New training features:", X_train_fe.shape[1])

print("Original testing features:", X_test.shape[1])
print("New testing features:", X_test_fe.shape[1])

Feature engineering applied successfully.
Original training features: 79
New training features: 84
Original testing features: 79
New testing features: 84


In [10]:
# Inspect engineered features

new_features = [
    "TotalSF",
    "HouseAge",
    "TotalBathrooms",
    "HasRemodeled",
    "TotalPorchSF"
]

display(X_train_fe[new_features].head())

print("\nSummary statistics:")
display(X_train_fe[new_features].describe())

,TotalSF,HouseAge,TotalBathrooms,HasRemodeled,TotalPorchSF
254,2628,53,2.0,0,250
1066,2370,16,2.5,1,40
638,1592,98,1.0,1,492
799,2499,70,2.5,1,264
380,2717,86,2.0,1,242



Summary statistics:


,TotalSF,HouseAge,TotalBathrooms,HasRemodeled,TotalPorchSF
count,1168.000000,1168.00000,1168.000000,1168.000000,1168.000000
mean,2583.476884,36.85274,2.222175,0.485445,186.584760
std,821.305316,30.70311,0.779502,0.500002,161.131777
min,334.000000,0.00000,1.000000,0.000000,0.000000
25%,2032.500000,7.00000,2.000000,0.000000,45.000000
50%,2494.000000,35.00000,2.000000,0.000000,168.000000
75%,3013.500000,55.00000,2.500000,1.000000,275.250000
max,11752.000000,136.00000,6.000000,1.000000,1027.000000


In [11]:
# Validate engineered features

print("Missing values in new features:")
display(X_train_fe[new_features].isna().sum())

print("\nNegative house ages:")
print((X_train_fe["HouseAge"] < 0).sum())

print("\nRemodel flag values:")
print(X_train_fe["HasRemodeled"].value_counts())

assert X_train_fe.shape[1] == X_train.shape[1] + 5
assert X_test_fe.shape[1] == X_test.shape[1] + 5
assert "SalePrice" not in X_train_fe.columns

print("\nFeature engineering validation completed.")

Missing values in new features:


TotalSF           0
HouseAge          0
TotalBathrooms    0
HasRemodeled      0
TotalPorchSF      0
dtype: int64


Negative house ages:
0

Remodel flag values:
HasRemodeled
0    601
1    567
Name: count, dtype: int64

Feature engineering validation completed.


## Block 3: Baseline Models with 5-Fold Cross-Validation

In this block, the original regression models will be evaluated using five-fold cross-validation.

The models include:

- Dummy Regressor (Mean Baseline)
- Linear Regression
- Ridge Regression
- Log-target Linear Regression

Cross-validation provides a more reliable performance estimate than a single train-test split.

The evaluation will use RMSE and R². RMSE will be reported as mean ± standard deviation across the five folds.

All preprocessing will be performed inside the model pipeline to prevent data leakage.

In [12]:
# Import additional tools for the modeling pipeline

from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import make_scorer

print("Additional modeling tools imported.")

Additional modeling tools imported.


In [13]:
# Create preprocessing function

def make_preprocessor(data):

    numeric_features = data.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = data.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    return preprocessor


print("Preprocessing function created successfully.")

Preprocessing function created successfully.


In [14]:
# Create a reusable complete pipeline

def make_baseline_pipeline(model, log_target=False):

    pipeline = Pipeline(steps=[
        (
            "feature_engineering",
            FunctionTransformer(
                engineer_features,
                validate=False
            )
        ),
        (
            "preprocessor",
            make_preprocessor(X_train_fe)
        ),
        ("model", model)
    ])

    if log_target:
        return TransformedTargetRegressor(
            regressor=pipeline,
            func=np.log1p,
            inverse_func=np.expm1
        )

    return pipeline


print("Complete modeling pipeline created.")

Complete modeling pipeline created.


In [15]:
# Define cross-validation scoring metrics

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

print("Scoring metrics configured.")

Scoring metrics configured.


In [16]:
# Define baseline models

baseline_models = {
    "Mean Baseline": DummyRegressor(strategy="mean"),

    "Linear Regression": LinearRegression(),

    "Ridge (alpha=0.1)": Ridge(alpha=0.1),

    "Log-target Linear Regression": LinearRegression()
}

print("Baseline models defined:")
for model_name in baseline_models:
    print("-", model_name)

Baseline models defined:
- Mean Baseline
- Linear Regression
- Ridge (alpha=0.1)
- Log-target Linear Regression


In [17]:
# Evaluate baseline models using cross-validation

baseline_cv_results = []

baseline_cv_models = {}

for model_name, model in baseline_models.items():

    print(f"\nEvaluating {model_name}...")

    log_target = model_name == "Log-target Linear Regression"

    pipeline = make_baseline_pipeline(
        model,
        log_target=log_target
    )

    start_time = time.time()

    cv_result = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    training_time = time.time() - start_time

    mean_rmse = -cv_result["test_RMSE"].mean()
    std_rmse = cv_result["test_RMSE"].std()

    mean_r2 = cv_result["test_R2"].mean()
    std_r2 = cv_result["test_R2"].std()

    baseline_cv_results.append({
        "Model": model_name,
        "Mean CV RMSE": mean_rmse,
        "Std CV RMSE": std_rmse,
        "Mean CV R2": mean_r2,
        "Std CV R2": std_r2,
        "Training Time (seconds)": training_time
    })

    baseline_cv_models[model_name] = pipeline

    print(f"Mean RMSE: {mean_rmse:,.2f}")
    print(f"Std RMSE:  {std_rmse:,.2f}")
    print(f"Mean R2:   {mean_r2:.4f}")
    print(f"Time:      {training_time:.2f} seconds")


print("\nBaseline cross-validation completed.")


Evaluating Mean Baseline...
Mean RMSE: 77,160.11
Std RMSE:  3,673.25
Mean R2:   -0.0016
Time:      8.40 seconds

Evaluating Linear Regression...
Mean RMSE: 36,257.55
Std RMSE:  7,443.19
Mean R2:   0.7717
Time:      0.69 seconds

Evaluating Ridge (alpha=0.1)...
Mean RMSE: 35,079.40
Std RMSE:  7,278.60
Mean R2:   0.7873
Time:      0.45 seconds

Evaluating Log-target Linear Regression...
Mean RMSE: 49,785.87
Std RMSE:  30,338.45
Mean R2:   0.4661
Time:      0.68 seconds

Baseline cross-validation completed.


In [18]:
# Create baseline comparison table

baseline_comparison = pd.DataFrame(baseline_cv_results)

baseline_comparison["CV RMSE (Mean ± Std)"] = (
    baseline_comparison["Mean CV RMSE"].map(lambda x: f"{x:,.2f}")
    + " ± "
    + baseline_comparison["Std CV RMSE"].map(lambda x: f"{x:,.2f}")
)

display(
    baseline_comparison[
        [
            "Model",
            "CV RMSE (Mean ± Std)",
            "Mean CV R2",
            "Training Time (seconds)"
        ]
    ].round(4)
)

,Model,CV RMSE (Mean ± Std),Mean CV R2,Training Time (seconds)
0,Mean Baseline,"77,160.11 ± 3,673.25",-0.0016,8.3989
1,Linear Regression,"36,257.55 ± 7,443.19",0.7717,0.6933
2,Ridge (alpha=0.1),"35,079.40 ± 7,278.60",0.7873,0.4490
3,Log-target Linear Regression,"49,785.87 ± 30,338.45",0.4661,0.6771


## Block 4: Hyperparameter Tuning with Ensemble Models

Ensemble learning combines multiple decision trees to improve predictive performance.

Two ensemble algorithms will be investigated:

**Random Forest Regression**

Random Forest uses bagging. It trains multiple decision trees on different samples of the training data and combines their predictions. This helps reduce variance and overfitting.

**XGBoost Regression**

XGBoost uses boosting. It builds trees sequentially, with each new tree attempting to correct errors made by previous trees.

Both models will be tuned using RandomizedSearchCV with five-fold cross-validation.

The search will explore multiple hyperparameters, and the best configurations will be selected based on mean cross-validation RMSE.

In [19]:
# Import XGBoost

try:
    import xgboost as xgb
    from xgboost import XGBRegressor

    print("XGBoost imported successfully.")
    print("XGBoost version:", xgb.__version__)

except ImportError:
    print("XGBoost is not installed.")
    print("Run: %pip install xgboost")

XGBoost imported successfully.
XGBoost version: 3.4.1


In [20]:
# Define Random Forest model

rf_pipeline = make_baseline_pipeline(
    RandomForestRegressor(
        random_state=42,
        n_jobs=1
    )
)

# Random Forest hyperparameter search space

rf_parameters = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.7, 1.0]
}

# Create RandomizedSearchCV

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_parameters,
    n_iter=20,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

print("Random Forest search configured.")
print("Hyperparameters:", len(rf_parameters))
print("Random combinations:", 20)
print("CV folds:", 5)

Random Forest search configured.
Hyperparameters: 5
Random combinations: 20
CV folds: 5


In [21]:
# Train and tune Random Forest

rf_start = time.time()

rf_search.fit(X_train, y_train)

rf_training_time = time.time() - rf_start

rf_best_index = rf_search.best_index_

rf_mean_rmse = -rf_search.cv_results_[
    "mean_test_RMSE"
][rf_best_index]

rf_std_rmse = rf_search.cv_results_[
    "std_test_RMSE"
][rf_best_index]

rf_mean_r2 = rf_search.cv_results_[
    "mean_test_R2"
][rf_best_index]

print("Random Forest tuning completed.")

print("\nBest parameters:")
print(rf_search.best_params_)

print(f"\nMean CV RMSE: {rf_mean_rmse:,.2f}")
print(f"Std CV RMSE: {rf_std_rmse:,.2f}")
print(f"Mean CV R2: {rf_mean_r2:.4f}")

print(f"Training time: {rf_training_time:.2f} seconds")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Random Forest tuning completed.

Best parameters:
{'model__n_estimators': 300, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 1.0, 'model__max_depth': 20}

Mean CV RMSE: 28,623.23
Std CV RMSE: 5,105.29
Mean CV R2: 0.8599
Training time: 450.34 seconds


In [22]:
# Define XGBoost model

xgb_pipeline = make_baseline_pipeline(
    XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=42,
        n_jobs=1,
        verbosity=0
    )
)

# XGBoost hyperparameter search space

xgb_parameters = {
    "model__n_estimators": [200, 300, 400, 600],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__min_child_weight": [1, 3, 5],
    "model__reg_lambda": [1, 5, 10]
}

# Create RandomizedSearchCV

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_parameters,
    n_iter=20,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

print("XGBoost search configured.")
print("Hyperparameters:", len(xgb_parameters))
print("Random combinations:", 20)
print("CV folds:", 5)

XGBoost search configured.
Hyperparameters: 7
Random combinations: 20
CV folds: 5


In [23]:
# Train and tune XGBoost

xgb_start = time.time()

xgb_search.fit(X_train, y_train)

xgb_training_time = time.time() - xgb_start

xgb_best_index = xgb_search.best_index_

xgb_mean_rmse = -xgb_search.cv_results_[
    "mean_test_RMSE"
][xgb_best_index]

xgb_std_rmse = xgb_search.cv_results_[
    "std_test_RMSE"
][xgb_best_index]

xgb_mean_r2 = xgb_search.cv_results_[
    "mean_test_R2"
][xgb_best_index]

print("XGBoost tuning completed.")

print("\nBest parameters:")
print(xgb_search.best_params_)

print(f"\nMean CV RMSE: {xgb_mean_rmse:,.2f}")
print(f"Std CV RMSE: {xgb_std_rmse:,.2f}")
print(f"Mean CV R2: {xgb_mean_r2:.4f}")

print(f"Training time: {xgb_training_time:.2f} seconds")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
XGBoost tuning completed.

Best parameters:
{'model__subsample': 1.0, 'model__reg_lambda': 1, 'model__n_estimators': 600, 'model__min_child_weight': 3, 'model__max_depth': 2, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.7}

Mean CV RMSE: 26,029.57
Std CV RMSE: 5,275.88
Mean CV R2: 0.8832
Training time: 63.77 seconds


## Block 5: Final Model Comparison and Holdout Testing

In this block, we will:

1. Refit the baseline models on the complete training set.
2. Use the best tuned Random Forest and XGBoost models.
3. Evaluate each model on the untouched 20% holdout test set.
4. Compare the final test metrics with the cross-validation results.
5. Decide which model is most suitable for deployment.

In [24]:
final_models = {
    "Mean Baseline": make_baseline_pipeline(
        DummyRegressor(strategy="mean")
    ),

    "Linear Regression": make_baseline_pipeline(
        LinearRegression()
    ),

    "Ridge (alpha=0.1)": make_baseline_pipeline(
        Ridge(alpha=0.1)
    ),

    "Log-target Linear Regression": make_baseline_pipeline(
        LinearRegression(),
        log_target=True
    ),

    "Random Forest (Tuned)": rf_search.best_estimator_,

    "XGBoost (Tuned)": xgb_search.best_estimator_
}

# Fit every model using training data only.
for name, model in final_models.items():
    model.fit(X_train, y_train)
    print(f"{name} fitted successfully.")

print("\nAll final models are ready for holdout evaluation.")

Mean Baseline fitted successfully.
Linear Regression fitted successfully.
Ridge (alpha=0.1) fitted successfully.
Log-target Linear Regression fitted successfully.
Random Forest (Tuned) fitted successfully.
XGBoost (Tuned) fitted successfully.

All final models are ready for holdout evaluation.


In [25]:
test_results = []

for name, model in final_models.items():

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    test_results.append({
        "Model": name,
        "Test MAE": mae,
        "Test RMSE": rmse,
        "Test R2": r2
    })

test_comparison = pd.DataFrame(test_results)

test_comparison = test_comparison.sort_values(
    by="Test RMSE"
).reset_index(drop=True)

print("Final Holdout Test Results:")
display(test_comparison.round(4))

Final Holdout Test Results:


,Model,Test MAE,Test RMSE,Test R2
0,Log-target Linear Regression,15035.2746,22749.0869,0.9325
1,XGBoost (Tuned),16247.4209,25850.9426,0.9129
2,Ridge (alpha=0.1),18293.8395,29200.1528,0.8888
3,Random Forest (Tuned),17376.9387,29536.4622,0.8863
4,Linear Regression,18329.9822,29543.9348,0.8862
5,Mean Baseline,62575.9265,87619.0345,-0.0009


In [26]:
# Prepare baseline CV results.
baseline_table = baseline_comparison.copy()

# Support either a Model column or model names stored in the index.
if "Model" in baseline_table.columns:
    baseline_table = baseline_table.set_index("Model")

# Match the tuned model results to the same table structure.
tuned_rows = pd.DataFrame([
    {
        "Mean CV RMSE": rf_mean_rmse,
        "Std CV RMSE": rf_std_rmse,
        "Mean CV R2": rf_mean_r2,
        "Training Time (seconds)": rf_training_time
    },
    {
        "Mean CV RMSE": xgb_mean_rmse,
        "Std CV RMSE": xgb_std_rmse,
        "Mean CV R2": xgb_mean_r2,
        "Training Time (seconds)": xgb_training_time
    }
], index=["Random Forest (Tuned)", "XGBoost (Tuned)"])

# Combine baseline and tuned models.
combined_cv = pd.concat([baseline_table, tuned_rows])

# Prepare holdout results for matching.
holdout_table = test_comparison.set_index("Model")

notes = {
    "Mean Baseline": "Reference model; performs poorly.",
    "Linear Regression": "Simple linear baseline.",
    "Ridge (alpha=0.1)": "Regularized linear model; strong baseline.",
    "Log-target Linear Regression": "Best holdout result, but unstable CV performance.",
    "Random Forest (Tuned)": "Tuned ensemble; slower search and weaker than XGBoost.",
    "XGBoost (Tuned)": "Best CV result; strong and consistent holdout performance."
}

comparison_rows = []

for model_name, row in combined_cv.iterrows():

    comparison_rows.append({
        "Model": model_name,
        "Mean CV RMSE": row["Mean CV RMSE"],
        "Std CV RMSE": row["Std CV RMSE"],
        "Mean CV R2": row["Mean CV R2"],
        "Training Time (seconds)": row["Training Time (seconds)"],
        "Holdout Test RMSE": holdout_table.loc[model_name, "Test RMSE"],
        "Note": notes[model_name]
    })

final_comparison = pd.DataFrame(comparison_rows)

final_comparison = final_comparison.sort_values(
    by="Mean CV RMSE"
).reset_index(drop=True)

display(final_comparison.round(3))

,Model,Mean CV RMSE,Std CV RMSE,Mean CV R2,Training Time (seconds),Holdout Test RMSE,Note
0,XGBoost (Tuned),26029.574,5275.881,0.883,63.773,25850.943,Best CV result; strong and consistent holdout ...
1,Random Forest (Tuned),28623.225,5105.287,0.860,450.342,29536.462,Tuned ensemble; slower search and weaker than ...
2,Ridge (alpha=0.1),35079.405,7278.603,0.787,0.449,29200.153,Regularized linear model; strong baseline.
3,Linear Regression,36257.554,7443.188,0.772,0.693,29543.935,Simple linear baseline.
4,Log-target Linear Regression,49785.872,30338.453,0.466,0.677,22749.087,"Best holdout result, but unstable CV performance."
5,Mean Baseline,77160.108,3673.253,-0.002,8.399,87619.035,Reference model; performs poorly.


## Deliverable 1: Feature Engineering Module

This file contains the reusable feature engineering function used in the regression pipelines.

In [27]:
%%writefile feature_engineering.py

import pandas as pd


def engineer_features(data):
    """
    Create domain-based features for house price prediction.

    Features:
    1. TotalSF: Basement + first floor + second floor area.
    2. HouseAge: Age of the house at the time of sale.
    3. TotalBathrooms: Full and half bathrooms combined.
    4. HasRemodeled: Whether the house was remodeled after construction.
    5. TotalPorchSF: Combined porch and deck area.
    """

    data = data.copy()

    data["TotalSF"] = (
        data["TotalBsmtSF"].fillna(0)
        + data["1stFlrSF"].fillna(0)
        + data["2ndFlrSF"].fillna(0)
    )

    data["HouseAge"] = data["YrSold"] - data["YearBuilt"]

    data["TotalBathrooms"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    data["HasRemodeled"] = (
        data["YearRemodAdd"] > data["YearBuilt"]
    ).astype(int)

    data["TotalPorchSF"] = (
        data["OpenPorchSF"].fillna(0)
        + data["EnclosedPorch"].fillna(0)
        + data["3SsnPorch"].fillna(0)
        + data["ScreenPorch"].fillna(0)
        + data["WoodDeckSF"].fillna(0)
    )

    return data

Writing feature_engineering.py


## Deliverable 2: Tuned Models Module

This file contains reusable preprocessing and model-building functions, baseline model definitions, and the hyperparameter search spaces used for Random Forest and XGBoost.

In [28]:
%%writefile tuned_models.py

import numpy as np

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV, KFold

from feature_engineering import engineer_features


def make_preprocessor(data):
    """Create preprocessing pipelines for numerical and categorical columns."""

    numeric_features = data.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = data.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])


def make_model_pipeline(model, training_features, log_target=False):
    """Build a complete feature engineering and preprocessing pipeline."""

    pipeline = Pipeline([
        (
            "feature_engineering",
            FunctionTransformer(engineer_features, validate=False)
        ),
        ("preprocessor", make_preprocessor(training_features)),
        ("model", model)
    ])

    if log_target:
        return TransformedTargetRegressor(
            regressor=pipeline,
            func=np.log1p,
            inverse_func=np.expm1
        )

    return pipeline


def get_baseline_models():
    """Return the baseline regression models."""

    return {
        "Mean Baseline": DummyRegressor(strategy="mean"),
        "Linear Regression": LinearRegression(),
        "Ridge (alpha=0.1)": Ridge(alpha=0.1),
        "Log-target Linear Regression": LinearRegression()
    }


def get_random_forest_parameters():
    """Return the Random Forest hyperparameter search space."""

    return {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.7, 1.0]
    }


def get_xgboost_parameters():
    """Return the XGBoost hyperparameter search space."""

    return {
        "model__n_estimators": [200, 300, 400, 600],
        "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4, 5, 6],
        "model__subsample": [0.7, 0.85, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 1.0],
        "model__min_child_weight": [1, 3, 5],
        "model__reg_lambda": [1, 5, 10]
    }


def tune_model(pipeline, parameters, X_train, y_train, cv, n_iter=20):
    """Tune a model using RandomizedSearchCV and RMSE."""

    scoring = {
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    }

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=parameters,
        n_iter=n_iter,
        scoring=scoring,
        refit="RMSE",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        return_train_score=False
    )

    search.fit(X_train, y_train)

    return search

Writing tuned_models.py


## Deliverable 3: Model Comparison Report

The final report documents the feature engineering process, five-fold cross-validation, hyperparameter tuning, final holdout evaluation, and deployment recommendation.

In [29]:
%%writefile MODEL_COMPARISON.md

# Feature Engineering & Tuned Ensembles for Regression

## 1. Project Overview

The objective of this project was to improve the house price prediction models developed in the previous regression task.

The work included:

- Creating at least four meaningful domain-based features.
- Re-evaluating baseline models using 5-fold cross-validation.
- Tuning Random Forest and XGBoost using RandomizedSearchCV.
- Comparing model performance, stability, and training time.
- Evaluating the final selected models on the untouched 20% holdout test set.

Dataset: Ames Housing Dataset

Target variable: SalePrice

Original dataset shape: 1460 rows and 86 columns.

The target, identifier column (`Id`), and five outlier indicator columns were excluded from the input features.

Final input features before engineering: 79.

Train/test split: 80% training and 20% testing.

Random state: 42.

Training samples: 1168.

Testing samples: 292.

## 2. Feature Engineering

Five new features were created using existing housing attributes.

| Engineered Feature | Description | Reason for Inclusion |
|---|---|---|
| TotalSF | Total basement, first-floor, and second-floor area | Represents the overall size of the property. |
| HouseAge | Year sold minus year built | Captures the age of the house at the time of sale. |
| TotalBathrooms | Full bathrooms plus half bathrooms counted as 0.5 | Represents the property's bathroom capacity. |
| HasRemodeled | Indicates whether the remodeling year is later than the construction year | Captures whether the property has been updated. |
| TotalPorchSF | Combined area of open porch, enclosed porch, three-season porch, screen porch, and wood deck | Represents additional outdoor living space. |

The dataset increased from 79 to 84 input features after feature engineering.

Missing area and bathroom values used in the engineered calculations were treated as zero where appropriate.

The feature engineering function was placed inside the modeling pipeline using FunctionTransformer. This ensured that feature creation was consistently applied during cross-validation and prediction.

## 3. Preprocessing

The following preprocessing steps were used:

### Numerical Features

- Missing values were replaced using median imputation.
- StandardScaler was used for feature scaling.

### Categorical Features

- Missing values were replaced using the most frequent value.
- OneHotEncoder was used to convert categorical values into numerical features.
- Unknown categories were handled using `handle_unknown="ignore"`.

A ColumnTransformer combined the numerical and categorical preprocessing pipelines.

All preprocessing operations were fitted within the training folds to avoid data leakage.

## 4. Five-Fold Cross-Validation

The baseline models were evaluated using five-fold cross-validation on the training data.

The evaluation metrics were:

- Mean RMSE
- Standard deviation of RMSE
- Mean R²

### Baseline Model Results

| Model | Mean CV RMSE | Std CV RMSE | Mean CV R² |
|---|---:|---:|---:|
| Mean Baseline | 77,160.11 | 3,673.25 | -0.0016 |
| Linear Regression | 36,257.55 | 7,443.19 | 0.7717 |
| Ridge (alpha=0.1) | 35,079.40 | 7,278.60 | 0.7873 |
| Log-target Linear Regression | 49,785.87 | 30,338.45 | 0.4661 |

Ridge Regression achieved the best mean CV RMSE among the baseline models.

The log-target Linear Regression model showed considerably higher variation across folds, suggesting less stable cross-validation performance.

## 5. Hyperparameter Tuning

RandomizedSearchCV was used to tune Random Forest and XGBoost.

Both searches used:

- 20 randomly selected parameter combinations.
- 5-fold cross-validation.
- RMSE as the refitting and selection metric.
- R² as an additional evaluation metric.
- Random state of 42.

Each search evaluated 100 model fits.

### Random Forest

Best parameters:

- n_estimators: 300
- max_depth: 20
- min_samples_split: 2
- min_samples_leaf: 2
- max_features: 1.0

Results:

- Mean CV RMSE: 28,623.23
- Standard deviation of RMSE: 5,105.29
- Mean CV R²: 0.8599
- Search training time: 450.34 seconds

### XGBoost

Best parameters:

- n_estimators: 600
- learning_rate: 0.1
- max_depth: 2
- subsample: 1.0
- colsample_bytree: 0.7
- min_child_weight: 3
- reg_lambda: 1

Results:

- Mean CV RMSE: 26,029.57
- Standard deviation of RMSE: 5,275.88
- Mean CV R²: 0.8832
- Search training time: 63.77 seconds

XGBoost achieved better cross-validation performance than Random Forest and completed its search considerably faster.

## 6. Overall Model Comparison

The following table combines cross-validation and final holdout results.

| Model | Mean CV RMSE | Std CV RMSE | Mean CV R² | Holdout RMSE |
|---|---:|---:|---:|---:|
| XGBoost (Tuned) | 26,029.57 | 5,275.88 | 0.8832 | 25,850.94 |
| Random Forest (Tuned) | 28,623.23 | 5,105.29 | 0.8599 | 29,536.46 |
| Ridge (alpha=0.1) | 35,079.40 | 7,278.60 | 0.7873 | 29,200.15 |
| Linear Regression | 36,257.55 | 7,443.19 | 0.7717 | 29,543.93 |
| Log-target Linear Regression | 49,785.87 | 30,338.45 | 0.4661 | 22,749.09 |
| Mean Baseline | 77,160.11 | 3,673.25 | -0.0016 | 87,619.03 |

### Training Time

| Model | Training/Search Time (seconds) |
|---|---:|
| XGBoost (Tuned) | 63.77 |
| Random Forest (Tuned) | 450.34 |
| Ridge (alpha=0.1) | 0.45 |
| Linear Regression | 0.69 |
| Log-target Linear Regression | 0.68 |
| Mean Baseline | 8.40 |

Note: Baseline times represent the recorded cross-validation evaluation time, whereas the Random Forest and XGBoost times include their complete randomized hyperparameter searches. These times are therefore not directly equivalent.

## 7. Final Holdout Test Evaluation

The original 20% test set contained 292 samples.

It was kept separate from cross-validation and hyperparameter tuning and evaluated once after model selection.

### Final Results

| Model | Test MAE | Test RMSE | Test R² |
|---|---:|---:|---:|
| Log-target Linear Regression | 15,035.27 | 22,749.09 | 0.9325 |
| XGBoost (Tuned) | 16,247.42 | 25,850.94 | 0.9129 |
| Ridge (alpha=0.1) | 18,293.84 | 29,200.15 | 0.8888 |
| Random Forest (Tuned) | 17,376.94 | 29,536.46 | 0.8863 |
| Linear Regression | 18,329.98 | 29,543.93 | 0.8862 |
| Mean Baseline | 62,575.93 | 87,619.03 | -0.0009 |

Log-target Linear Regression achieved the lowest holdout RMSE and MAE, as well as the highest holdout R².

XGBoost achieved the second-best holdout RMSE and maintained strong performance across both cross-validation and holdout evaluation.

## 8. Model Selection and Verdict

XGBoost is the recommended model to ship.

The reasons are:

1. It achieved the lowest mean cross-validation RMSE among all evaluated models.
2. It achieved the highest mean cross-validation R².
3. Its holdout RMSE was close to its cross-validation RMSE.
4. It performed better than the tuned Random Forest model.
5. Its tuning process took considerably less time than Random Forest.

However, log-target Linear Regression achieved the best result on the final holdout test set.

Its holdout RMSE was approximately 22,749, compared with approximately 25,851 for XGBoost.

Despite this result, its cross-validation performance was considerably weaker and its standard deviation was much higher. Therefore, the single holdout result is not sufficient evidence that it will consistently outperform XGBoost on other data.

The improvement from feature engineering and ensemble tuning is practically meaningful compared with the mean baseline and the earlier linear models. However, the additional complexity of XGBoost should be considered alongside its improved cross-validation performance.

No formal statistical significance claim is made because repeated or nested cross-validation was not performed.

## 9. Conclusion

This project demonstrated how domain-based feature engineering and ensemble learning can improve regression performance.

Five new features were engineered, and preprocessing was integrated into pipelines to reduce data leakage.

RandomizedSearchCV was used to tune Random Forest and XGBoost over multiple hyperparameters.

XGBoost provided the strongest and most dependable cross-validation performance and is the recommended model for deployment.

Log-target Linear Regression achieved the best result on the single untouched holdout test set, highlighting the importance of comparing cross-validation and final test performance rather than relying on one metric alone.

The project successfully completed feature engineering, model tuning, model comparison, and final holdout evaluation.

Writing MODEL_COMPARISON.md


In [30]:
from pathlib import Path

required_files = [
    "feature_engineering.py",
    "tuned_models.py",
    "MODEL_COMPARISON.md"
]

for filename in required_files:
    path = Path(filename)

    if path.exists():
        print(f"FOUND: {filename} ({path.stat().st_size} bytes)")
    else:
        print(f"MISSING: {filename}")

FOUND: feature_engineering.py (1263 bytes)
FOUND: tuned_models.py (3683 bytes)
FOUND: MODEL_COMPARISON.md (8758 bytes)
